# 📓 Notebook 01 — Data Understanding
## Student Placement Prediction System
### Author: Sachin — Data Collection, Cleaning & Preprocessing

This notebook demonstrates:
- Loading the CSV dataset
- Schema inspection (shape, dtypes, sample)
- Missing value detection and handling
- Duplicate detection and removal
- Outlier identification (IQR method)
- Target variable encoding
- Categorical encoding & numerical scaling
- Stratified train/test split
- Pipeline construction

> **Important**: Before running this notebook, place your downloaded CSV in:
> `data/college_student_placement.csv`

In [ ]:
import sys, os
# Make src/ importable from the notebook
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

print('Libraries loaded ✓')

## Step 1: Load the Dataset

In [ ]:
from data_preprocessing import load_dataset, display_schema

df = load_dataset()   # reads data/college_student_placement.csv
display_schema(df)

In [ ]:
# Visual summary
print('\nColumn data types:')
print(df.dtypes)
print('\nDescriptive statistics:')
df.describe().round(2)

## Step 2: Missing Value Analysis

In [ ]:
from data_preprocessing import report_missing_values, handle_missing_values

missing_summary = report_missing_values(df)

# Visual: missing value heatmap
if not missing_summary.empty:
    fig, ax = plt.subplots(figsize=(10, 3))
    sns.heatmap(df.isnull().T, cbar=False, cmap='Reds', ax=ax)
    ax.set_title('Missing Value Map (Red = Missing)')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values — no heatmap needed.')

df = handle_missing_values(df)
print('\nAfter imputation — missing values:', df.isnull().sum().sum())

## Step 3: Duplicate Detection

In [ ]:
from data_preprocessing import report_duplicates
df = report_duplicates(df)
print(f'Dataset shape after deduplication: {df.shape}')

## Step 4: Target Variable Encoding

In [ ]:
from data_preprocessing import detect_target_column, map_target, drop_id_columns

df = drop_id_columns(df)
target_col = detect_target_column(df)
print(f'Target column: {target_col}')
print(f'Unique values before mapping: {df[target_col].unique()}')

df = map_target(df, target_col)
print(f'\nClass distribution (0=Not Placed, 1=Placed):')
print(df[target_col].value_counts())

# Pie chart
fig, ax = plt.subplots(figsize=(5, 5))
counts = df[target_col].value_counts()
ax.pie(counts, labels=['Placed (1)', 'Not Placed (0)'],
       colors=['#2ECC71', '#E74C3C'], autopct='%1.1f%%',
       startangle=90, shadow=True)
ax.set_title('Class Distribution')
plt.show()

## Step 5: Outlier Detection (IQR)

In [ ]:
from data_preprocessing import identify_column_types, detect_outliers_iqr

numeric_features, categorical_features = identify_column_types(df, target_col)
outlier_counts = detect_outliers_iqr(df, numeric_features)

# Visual: outlier count bar chart
oc_series = pd.Series({k: v for k, v in outlier_counts.items() if v > 0})
if not oc_series.empty:
    fig, ax = plt.subplots(figsize=(8, 3))
    oc_series.sort_values(ascending=False).plot(kind='bar', color='#E74C3C', ax=ax)
    ax.set_title('Outlier Count per Feature (IQR method)')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('No outliers detected.')

## Step 6: Pipeline Construction & Train/Test Split

In [ ]:
from data_preprocessing import split_data, build_preprocessor

X_train, X_test, y_train, y_test = split_data(df, target_col)

preprocessor = build_preprocessor(numeric_features, categorical_features)
preprocessor.fit(X_train)   # Fit ONLY on training data!

X_train_transformed = preprocessor.transform(X_train)
X_test_transformed  = preprocessor.transform(X_test)

print(f'\nTransformed training set shape: {X_train_transformed.shape}')
print(f'Transformed test set shape     : {X_test_transformed.shape}')
print('\n✓ Preprocessing complete. The test set has NOT been used for fitting.')

## Summary

| Step | Action | Outcome |
|------|--------|---------|
| Load | `load_dataset()` | DataFrame of shape (N, M) |
| Missing values | Median/mode imputation | No NaN values remain |
| Duplicates | Removed | Clean dataset |
| Target encoding | Yes/No → 1/0 | Binary integer target |
| ID columns | Dropped College_ID | Removed non-predictive identifier |
| Outliers | Detected (IQR) | Reported, not removed |
| Split | Stratified 80/20 | X_train, X_test, y_train, y_test |
| Pipeline | ColumnTransformer | Scaler + Encoder, fitted on train only |